# 07 · Are the attacking-value rankings dependable?

Use the complete cached run from notebook 06. No downloads or new packages are required for the calculations. Optional figures use matplotlib.

**Questions:** Do rankings change with grid resolution or minutes eligibility? Does one exceptional match dominate a player's total? What do the leading players' valued actions actually look like?

The primary specification remains the frozen **12×8, 2018-trained model**. We compare predeclared 8×6 and 16×12 variants and 180/300/450-minute eligibility thresholds. These are exploratory sensitivity analyses on 2022, not new evidence of predictive accuracy. Do not select a preferred configuration by whichever yields an attractive player ranking. If these findings guide model changes, evaluate those changes on another untouched cohort.

All results concern signed value from successful selected open-play movements, not overall player impact. See notebook 06 for the selected play-pattern definition and failed-action limitations.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


In [ ]:
import gzip
import hashlib
import math
import random
import statistics
from datetime import datetime,timezone

REPORT_ROOT = ROOT / "reports/xt_robustness"
REPORT_ROOT.mkdir(parents=True,exist_ok=True)
BASELINE_GRID=(12,8)
GRIDS=[(8,6),(12,8),(16,12)]
MINUTE_THRESHOLDS=[180,300,450]
NX,NY=BASELINE_GRID
MIN_ZONE_ACTIONS=20
BOOTSTRAP_REPLICATES=2000
SEED=42
run=json.loads((ROOT / "reports/xt_run_manifest.json").read_text(encoding="utf-8"))
source=json.loads((ROOT / "sources.lock.json").read_text(encoding="utf-8"))
training_root=ROOT / "data/xt_training_2018"
training_lock=json.loads((training_root / "sources.lock.json").read_text(encoding="utf-8"))
if len(run["training_match_ids"])!=64 or len(run["application_match_ids"])!=64:
    raise ValueError("Run notebook 06 on both complete tournaments first.")
if run["grid"]!=[12,8] or run["allowed_patterns"]!=["From Counter","Regular Play"]:
    raise ValueError("Baseline settings differ. Review this notebook before comparing results.")
if run["source_revision"]!=source["revisions"]["statsbomb"] or training_lock["revision"]!=run["source_revision"]:
    raise ValueError("Source revisions differ.")
train_ids=run["training_match_ids"]
eval_ids=run["application_match_ids"]
assert set(train_ids).isdisjoint(eval_ids)

def load_verified(path,record,compressed=False):
    opener=gzip.open if compressed else open
    with opener(path,"rb") as stream:
        data=stream.read()
    if hashlib.sha256(data).hexdigest()!=record["sha256"]:
        raise ValueError(f"Checksum mismatch: {path}")
    return json.loads(data)

train_catalog=load_verified(training_root / "data/matches/43/3.json.gz",training_lock["files"]["data/matches/43/3.json"],True)
eval_catalog=load_verified(ROOT / "data/raw/statsbomb/matches/43_106.json",source["files"]["statsbomb/matches/43_106.json"])
train_dates={str(m["match_id"]):m["match_date"] for m in train_catalog}
match_metadata={str(m["match_id"]):m for m in eval_catalog}
assert max(train_dates[mid] for mid in train_ids)<min(match_metadata[mid]["match_date"] for mid in eval_ids)

def read_rows(path):
    with path.open(newline="",encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

saved_tournament=read_rows(ROOT / "reports/xt_tournament_players.csv")
saved_matches=read_rows(ROOT / "reports/xt_player_match.csv")
NUMERIC_FIELDS=["minutes","pass_xt","carry_xt","movement_xt","movement_xt_per90","npxg","nonpen_goals","scored_moves","unscored_moves"]
for rows in (saved_tournament,saved_matches):
    for r in rows:
        r["team_id"]=int(r["team_id"]); r["player_id"]=int(r["player_id"])
        for f in NUMERIC_FIELDS:
            r[f]=float(r[f]) if r[f]!="" else None
if {r["match_id"] for r in saved_matches}!=set(eval_ids):
    raise ValueError("Saved player-match table does not match run manifest.")
minutes={(r["match_id"],r["team_id"],r["player_id"]):r for r in saved_matches}
print("Verified cohort: 64 training matches from 2018, 64 application matches from 2022.")


## 1. Use the same xT definitions for every grid

Only grid resolution changes. The action scope, source revision, goal outcomes, failure treatment and signed attribution remain fixed. These functions are copied from notebook 06 so this file runs without executing another notebook.


In [ ]:
import math
from collections import Counter,defaultdict
ALLOWED_PATTERNS={"Regular Play","From Counter"}

def zone(loc,nx=NX,ny=NY):
    if not isinstance(loc,(list,tuple)) or len(loc)<2:
        return None
    x,y=loc[:2]
    if not all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v) for v in (x,y)):
        return None
    if not (0<=x<=120 and 0<=y<=80):
        return None
    return min(ny-1,int(y/80*ny))*nx+min(nx-1,int(x/120*nx))

def extract_actions(events,match_id,nx=NX,ny=NY):
    audit=Counter(events=len(events))
    ids=[e["id"] for e in events]
    if len(ids)!=len(set(ids)):
        raise ValueError(f"Duplicate event IDs: {match_id}")
    actions=[]
    for e in events:
        kind=e["type"]["name"]
        if e["period"]==5 or kind not in {"Pass","Carry","Shot"}:
            continue
        if e.get("play_pattern",{}).get("name") not in ALLOWED_PATTERNS:
            audit["excluded_pattern"]+=1
            continue
        if kind=="Shot" and e["shot"]["type"]["name"]=="Penalty":
            audit["excluded_penalty"]+=1
            continue
        audit["candidate_actions"]+=1
        start=e.get("location")
        start_zone=zone(start,nx,ny)
        if start_zone is None or "player" not in e:
            audit["invalid_start_or_actor"]+=1
            continue
        successful=kind=="Carry" or (kind=="Pass" and "outcome" not in e["pass"])
        end=e.get(kind.lower(),{}).get("end_location") if kind!="Shot" else None
        end_zone=zone(end,nx,ny) if successful else None
        if successful and end_zone is None:
            audit["invalid_successful_end"]+=1
            continue
        actions.append(dict(match_id=str(match_id),event_id=e["id"],event_index=e.get("index"),period=e["period"],timestamp=e["timestamp"],
            team_id=e["team"]["id"],team=e["team"]["name"],player_id=e["player"]["id"],player=e["player"]["name"],
            kind=kind,start_zone=start_zone,end_zone=end_zone,start_x=start[0],start_y=start[1],
            end_x=end[0] if successful else None,end_y=end[1] if successful else None,successful=successful,
            goal=kind=="Shot" and e["shot"]["outcome"]["name"]=="Goal"))
        audit["retained_actions"]+=1
    return actions,dict(audit)

def fit_xt(actions,nx=NX,ny=NY,tolerance=1e-10,max_iterations=10000):
    size=nx*ny
    starts=[0]*size
    goals=[0]*size
    shots=[0]*size
    failed_passes=[0]*size
    transitions=[Counter() for _ in range(size)]
    for a in actions:
        i=a["start_zone"]
        starts[i]+=1
        if a["kind"]=="Shot":
            shots[i]+=1
            goals[i]+=int(a["goal"])
        elif a["successful"]:
            transitions[i][a["end_zone"]]+=1
        else:
            failed_passes[i]+=1
    if not any(goals):
        raise ValueError("Training actions contain no goals; cannot fit a meaningful goal-based surface.")
    values=[0.0]*size
    for iteration in range(1,max_iterations+1):
        updated=[(goals[i]+sum(count*values[j] for j,count in transitions[i].items()))/starts[i] if starts[i] else 0.0 for i in range(size)]
        residual=max(abs(a-b) for a,b in zip(updated,values))
        values=updated
        if residual<tolerance:
            break
    else:
        raise RuntimeError("xT iteration did not converge; inspect transitions and settings.")
    assert all(0<=v<=1+1e-9 for v in values)
    return dict(nx=nx,ny=ny,values=values,start_counts=starts,goals=goals,shot_counts=shots,failed_passes=failed_passes,
                transition_counts=[dict(c) for c in transitions],iterations=iteration,residual=residual)

def rate_movement(action,model):
    if action["kind"] not in {"Pass","Carry"} or not action["successful"]:
        return None
    i,j=action["start_zone"],action["end_zone"]
    if not model["start_counts"][i] or not model["start_counts"][j]:
        return None  # Unsupported zones are missing, not false zero-valued estimates.
    return model["values"][j]-model["values"][i]


## 2. Refit on 2018 and apply to 2022

Training inputs are verified against the compressed training manifest. Application events are verified against the main source manifest. Each match is read separately to limit memory use. All fit calls below receive only 2018 actions.

A comparison model may have more sparse cells or unscorable movements. Those diagnostics remain visible. Do not interpret the 20-action flag as a measure of statistical certainty; 20 actions can contain very little goal information.


In [ ]:
training_actions={grid:[] for grid in GRIDS}
for mid in train_ids:
    remote=f"data/events/{mid}.json"
    events=load_verified(training_root / (remote+".gz"),training_lock["files"][remote],True)
    for grid in GRIDS:
        actions,_=extract_actions(events,mid,*grid)
        training_actions[grid].extend(actions)
models={}
for grid in GRIDS:
    models[grid]=fit_xt(training_actions[grid],*grid)
    print("Fitted",grid,"| actions",len(training_actions[grid]),"| goals",sum(models[grid]["goals"]))
del training_actions

scenario_matches={grid:{} for grid in GRIDS}
diagnostics={}
for grid in GRIDS:
    diagnostics[grid]=dict(unscored_movements=0,low_support_movements=0,scored_movements=0,
        unobserved_training_cells=sum(n==0 for n in models[grid]["start_counts"]),
        sparse_training_cells=sum(n<MIN_ZONE_ACTIONS for n in models[grid]["start_counts"]))
    for key,exposure in minutes.items():
        scenario_matches[grid][key]=dict(match_id=key[0],team_id=key[1],player_id=key[2],team=exposure["team"],player=exposure["player"],
            minutes=exposure["minutes"],pass_xt=0.0,carry_xt=0.0,movement_xt=0.0,scored_moves=0,unscored_moves=0,low_support_moves=0)
for mid in eval_ids:
    relative=f"statsbomb/events/{mid}.json"
    events=load_verified(ROOT / "data/raw" / relative,source["files"][relative])
    for grid in GRIDS:
        actions,_=extract_actions(events,mid,*grid)
        for a in actions:
            if a["kind"] not in {"Pass","Carry"} or not a["successful"]:
                continue
            key=(mid,a["team_id"],a["player_id"])
            if key not in scenario_matches[grid]:
                raise ValueError("Playing action missing from saved exposure table")
            row=scenario_matches[grid][key]
            value=rate_movement(a,models[grid])
            if value is None:
                row["unscored_moves"]+=1
                diagnostics[grid]["unscored_movements"]+=1
                continue
            low=min(models[grid]["start_counts"][a["start_zone"]],models[grid]["start_counts"][a["end_zone"]])<MIN_ZONE_ACTIONS
            row["scored_moves"]+=1
            row["low_support_moves"]+=int(low)
            row["pass_xt" if a["kind"]=="Pass" else "carry_xt"]+=value
            row["movement_xt"]+=value
            diagnostics[grid]["scored_movements"]+=1
            diagnostics[grid]["low_support_movements"]+=int(low)

scenario_players={}
for grid,match_rows in scenario_matches.items():
    players={}
    for r in match_rows.values():
        key=(r["team_id"],r["player_id"])
        total=players.setdefault(key,dict(team_id=key[0],player_id=key[1],team=r["team"],player=r["player"],
            minutes=0.0,matches=0,pass_xt=0.0,carry_xt=0.0,movement_xt=0.0,scored_moves=0,unscored_moves=0,low_support_moves=0))
        total["matches"]+=1
        for f in ["minutes","pass_xt","carry_xt","movement_xt","scored_moves","unscored_moves","low_support_moves"]:
            total[f]+=r[f]
    for r in players.values():
        r["movement_xt_per90"]=90*r["movement_xt"]/r["minutes"] if r["minutes"]>0 else None
    scenario_players[grid]=players
    print(grid,diagnostics[grid])

# Reproduction must pass before treating alternate grids as a meaningful comparison.
original={(r["team_id"],r["player_id"]):r for r in saved_tournament}
assert set(original)==set(scenario_players[BASELINE_GRID])
for key,row in scenario_players[BASELINE_GRID].items():
    for f in ["minutes","movement_xt","pass_xt","carry_xt","scored_moves","unscored_moves"]:
        assert math.isclose(row[f],original[key][f],rel_tol=1e-8,abs_tol=1e-9),(key,f)
print("12×8 baseline exactly reproduced within floating-point tolerance.")


## 3. Compare rankings on equal eligibility rules

We rank movement xT **per 90** after each minutes filter. Every displayed ranking also requires no unscorable movements and at least one scored movement. Scores for other players remain saved.

Compare grid variants at the **same minutes threshold**. Spearman correlation is calculated on common eligible players using average ranks for ties. Top-10 overlap is the number of shared players, not a significance test. Near-identical scores can swap ranks without a meaningful change in contribution.


In [ ]:
def average_ranks(values):
    ordered=sorted(values,key=lambda k:-values[k])
    result={}; i=0
    while i<len(ordered):
        j=i+1
        while j<len(ordered) and values[ordered[j]]==values[ordered[i]]:
            j+=1
        rank=(i+1+j)/2
        for key in ordered[i:j]: result[key]=rank
        i=j
    return result

def spearman(left,right):
    keys=sorted(set(left)&set(right))
    if len(keys)<3: return None
    a=average_ranks({k:left[k] for k in keys}); b=average_ranks({k:right[k] for k in keys})
    mean_a=statistics.mean(a.values()); mean_b=statistics.mean(b.values())
    numerator=sum((a[k]-mean_a)*(b[k]-mean_b) for k in keys)
    denominator=math.sqrt(sum((a[k]-mean_a)**2 for k in keys)*sum((b[k]-mean_b)**2 for k in keys))
    return numerator/denominator if denominator else None

assert average_ranks({"a":2,"b":2,"c":1})=={"a":1.5,"b":1.5,"c":3.0}
assert math.isclose(spearman({"a":1,"b":2,"c":3},{"a":3,"b":2,"c":1}),-1)
rankings={}; sensitivity_rows=[]
for grid in GRIDS:
    for minimum in MINUTE_THRESHOLDS:
        eligible={k:r["movement_xt_per90"] for k,r in scenario_players[grid].items()
                  if r["minutes"]>=minimum and r["unscored_moves"]==0 and r["scored_moves"]>0}
        ranks=average_ranks(eligible)
        rankings[(grid,minimum)]=dict(scores=eligible,ranks=ranks,
            top10=sorted(eligible,key=lambda k:(-eligible[k],k))[:10])
        for key,rank in ranks.items():
            r=scenario_players[grid][key]
            sensitivity_rows.append(dict(grid=f"{grid[0]}x{grid[1]}",min_minutes=minimum,rank=rank,**r))
comparisons=[]
for minimum in MINUTE_THRESHOLDS:
    reference=rankings[(BASELINE_GRID,minimum)]
    for grid in GRIDS:
        current=rankings[(grid,minimum)]
        common=set(reference["scores"])&set(current["scores"])
        comparison=dict(grid=f"{grid[0]}x{grid[1]}",min_minutes=minimum,eligible_players=len(current["scores"]),common_players=len(common),
            spearman_vs_baseline=spearman(reference["scores"],current["scores"]),
            top10_overlap=len(set(reference["top10"])&set(current["top10"])))
        comparisons.append(comparison)
        print(comparison)


## 4. Match consistency and illustrative case studies

Default case studies are the three highest **baseline tournament movement-xT totals**. Their selection is retrospective and illustrative, not evidence that the model discovered them prospectively. You can replace `FOCUS_KEYS` with `(team_id, player_id)` pairs from the saved table.

We show match totals, minutes, positive/negative match counts, and the largest-match share of **positive** contribution. This share uses positive contributions in the denominator because signed net totals can approach zero or become negative.

Match-resampling intervals below sample each player's observed appearances with replacement and recompute total xT / sampled minutes × 90. They hold the model fixed, assume exchangeable appearances, and do not capture model uncertainty, opponent selection, shared-match dependence, tactical changes or unseen opponents. With only a handful of appearances, these are descriptive sensitivity intervals, not calibrated confidence intervals or proof of repeatable ability. Per-90 rates for short substitutes can be especially unstable.


In [ ]:
baseline=scenario_players[BASELINE_GRID]
leaders=sorted(baseline,key=lambda k:(-baseline[k]["movement_xt"],k))[:3]
FOCUS_KEYS=leaders

def percentile(values,p):
    values=sorted(values)
    index=(len(values)-1)*p
    lo=math.floor(index); hi=math.ceil(index)
    return values[lo]+(values[hi]-values[lo])*(index-lo)

def match_resampling_interval(appearances,seed):
    rng=random.Random(seed)
    values=[]
    for _ in range(BOOTSTRAP_REPLICATES):
        sample=[rng.choice(appearances) for _ in appearances]
        duration=sum(r["minutes"] for r in sample)
        if duration>0: values.append(90*sum(r["movement_xt"] for r in sample)/duration)
    return (percentile(values,.025),percentile(values,.975)) if values else (None,None)

case_summaries=[]
case_matches=[]
for index,key in enumerate(FOCUS_KEYS):
    player=baseline[key]
    appearances=sorted([r for k,r in scenario_matches[BASELINE_GRID].items() if k[1:]==key and r["minutes"]>0],
                       key=lambda r:(match_metadata[r["match_id"]]["match_date"],r["match_id"]))
    positive=sum(max(r["movement_xt"],0) for r in appearances)
    low,high=match_resampling_interval(appearances,SEED+index)
    summary=dict(player=player["player"],team=player["team"],team_id=key[0],player_id=key[1],matches=len(appearances),minutes=player["minutes"],
        movement_xt=player["movement_xt"],pass_xt=player["pass_xt"],carry_xt=player["carry_xt"],movement_xt_per90=player["movement_xt_per90"],
        positive_matches=sum(r["movement_xt"]>0 for r in appearances),negative_matches=sum(r["movement_xt"]<0 for r in appearances),
        largest_match_share_of_positive_xt=max((max(r["movement_xt"],0) for r in appearances),default=0)/positive if positive else None,
        resampling_lower=low,resampling_upper=high,
        scenario_appearances=sum(key in result["ranks"] for result in rankings.values()))
    rates_without_match=[]
    for r in appearances:
        duration=player["minutes"]-r["minutes"]
        if duration>0: rates_without_match.append(90*(player["movement_xt"]-r["movement_xt"])/duration)
        meta=match_metadata[r["match_id"]]
        row=dict(r,opponent=meta["away_team"]["away_team_name"] if r["team_id"]==meta["home_team"]["home_team_id"] else meta["home_team"]["home_team_name"],match_date=meta["match_date"])
        case_matches.append(row)
    summary["leave_one_match_out_rate_min"]=min(rates_without_match) if rates_without_match else None
    summary["leave_one_match_out_rate_max"]=max(rates_without_match) if rates_without_match else None
    case_summaries.append(summary)
    print("\n",summary)
    for r in appearances:
        print(r["match_id"],f"{r['minutes']:.1f} min | pass {r['pass_xt']:+.4f} | carry {r['carry_xt']:+.4f} | total {r['movement_xt']:+.4f}")


## 5. Explain the largest actions, not just the score

Read event ID, period, timestamp and pitch coordinates to locate actions for manual review. Each map depicts the five highest-valued movements for one focus player, across their matches, with match IDs attached. They are different match contexts overlaid on a standardized pitch, not one attacking sequence. The selection highlights positive contributions; it does not show every loss or all negative movements.

Shot/goal statistics are separate evidence. No shooting score is added to movement xT. The maps cannot show defenders or off-ball runs and do not replace video.


In [ ]:
focus_actions=[]
for a in read_rows(ROOT / "reports/xt_action_values.csv"):
    key=(int(a["team_id"]),int(a["player_id"]))
    if key not in FOCUS_KEYS or a["xt"]=="": continue
    if a["match_id"] not in eval_ids: raise ValueError("Action table contains another cohort")
    for f in ["xt","start_x","start_y","end_x","end_y"]: a[f]=float(a[f])
    a["player_id"]=int(a["player_id"]); a["team_id"]=int(a["team_id"])
    focus_actions.append(a)
action_reviews=[]
for key in FOCUS_KEYS:
    assert math.isclose(sum(a["xt"] for a in focus_actions if (a["team_id"],a["player_id"])==key),baseline[key]["movement_xt"],rel_tol=1e-8,abs_tol=1e-9), "Saved action values do not reconcile with baseline"
    selected_actions=sorted([a for a in focus_actions if (a["team_id"],a["player_id"])==key],key=lambda a:-a["xt"])[:5]
    print("\n",baseline[key]["player"])
    for rank,a in enumerate(selected_actions,1):
        action_reviews.append(dict(action_rank=rank,**a))
        print(a["match_id"],a["period"],a["timestamp"],a["kind"],f"{a['xt']:+.4f}",a["event_id"])


## 6. Optional portfolio figures

Install matplotlib in this kernel if needed (`%pip install matplotlib`). Figures show specification sensitivity, the case-study match breakdowns, and annotated high-value actions. Grid columns are deliberately ordered 8×6, 12×8, 16×12. Missing eligibility is shown as a gap, not assigned a worst rank.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt=None
    print("Plots skipped. Run %pip install matplotlib to enable them.")
if plt is not None:
    figures=REPORT_ROOT / "figures"
    figures.mkdir(exist_ok=True)
    fig,axes=plt.subplots(1,3,figsize=(15,5),sharey=True,constrained_layout=True)
    for ax,minimum in zip(axes,MINUTE_THRESHOLDS):
        for key in FOCUS_KEYS:
            ranks=[rankings[(grid,minimum)]["ranks"].get(key,float("nan")) for grid in GRIDS]
            ax.plot(range(len(GRIDS)),ranks,marker="o",label=baseline[key]["player"])
        ax.set_xticks(range(len(GRIDS)),[f"{g[0]}×{g[1]}" for g in GRIDS])
        ax.set_title(f"At least {minimum} elapsed minutes")
        ax.set_xlabel("Training grid")
        ax.grid(alpha=.2)
    axes[0].invert_yaxis()
    max_focus_rank=max(result["ranks"][key] for result in rankings.values() for key in FOCUS_KEYS if key in result["ranks"])
    axes[0].set_ylim(max_focus_rank+2,.5)
    axes[0].set_yticks([t for t in [1,5,10,20,30,40,50] if t<=max_focus_rank+2])
    axes[0].set_ylabel("Movement xT/90 rank (1 is highest)")
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="outside lower center",ncol=1,fontsize=8)
    fig.suptitle("Exploratory rank sensitivity; gaps indicate ineligibility")
    fig.savefig(figures / "rank_sensitivity.png",dpi=160)
    plt.show()
    fig,axes=plt.subplots(len(FOCUS_KEYS),1,figsize=(12,3.6*len(FOCUS_KEYS)),squeeze=False,constrained_layout=True)
    for ax,key in zip(axes[:,0],FOCUS_KEYS):
        rows=sorted([r for r in case_matches if (r["team_id"],r["player_id"])==key],key=lambda r:(r["match_date"],r["match_id"]))
        xs=list(range(len(rows)))
        ax.bar([x-.18 for x in xs],[r["pass_xt"] for r in rows],width=.35,label="Passing")
        ax.bar([x+.18 for x in xs],[r["carry_xt"] for r in rows],width=.35,label="Carrying")
        ax.axhline(0,color="gray",lw=.8)
        ax.set_xticks(xs,[r["opponent"]+f"\n{r['minutes']:.0f} min" for r in rows],fontsize=8)
        ax.set_ylabel("Signed movement xT")
        ax.set_title(baseline[key]["player"])
        ax.legend(fontsize=8)
    fig.savefig(figures / "case_match_contributions.png",dpi=160)
    plt.show()
    fig,axes=plt.subplots(1,len(FOCUS_KEYS),figsize=(6*len(FOCUS_KEYS),5),squeeze=False,constrained_layout=True)
    for ax,key in zip(axes[0],FOCUS_KEYS):
        ax.set_xlim(0,120); ax.set_ylim(80,0); ax.set_aspect("equal")
        ax.plot([0,120,120,0,0],[0,0,80,80,0],color="gray",lw=1)
        ax.plot([60,60],[0,80],color="gray",lw=.8)
        ax.plot([120,102,102,120],[18,18,62,62],color="gray",lw=.8)
        largest=sorted([a for a in focus_actions if (a["team_id"],a["player_id"])==key],key=lambda a:-a["xt"])[:5]
        for rank,a in enumerate(largest,1):
            color="tab:blue" if a["kind"]=="Pass" else "tab:green"
            ax.annotate("",xy=(a["end_x"],a["end_y"]),xytext=(a["start_x"],a["start_y"]),arrowprops=dict(arrowstyle="->",lw=1.7,color=color,alpha=.85))
            ax.text(a["start_x"],a["start_y"],str(rank),fontsize=8,bbox=dict(facecolor="white",edgecolor="none",alpha=.8))
        ax.set_title(baseline[key]["player"]+"\nTop 5 positive movements across matches",fontsize=9)
        ax.set_xlabel("Attack to right; blue passes, green carries",fontsize=8)
    fig.savefig(figures / "case_top_actions.png",dpi=160)
    plt.show()


## 7. Save the analysis and a portfolio findings draft

The report distinguishes reproduced baseline results, exploratory ranking changes and case-study observations. Read it alongside sparse-zone diagnostics and action/video review. Neither rank correlation nor resampling validates the model's scoring probabilities. The resampling intervals exclude training-model uncertainty and are especially limited with few appearances.


In [ ]:
def write_csv(name,rows):
    if not rows: raise ValueError(f"Empty output: {name}")
    with (REPORT_ROOT / name).open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
write_csv("ranking_scenarios.csv",sensitivity_rows)
write_csv("grid_comparisons.csv",comparisons)
write_csv("case_summaries.csv",case_summaries)
write_csv("case_match_contributions.csv",case_matches)
write_csv("case_action_values.csv",focus_actions)
write_csv("case_top_action_reviews.csv",action_reviews)
manifest=dict(created_at=datetime.now(timezone.utc).isoformat(),source_revision=run["source_revision"],
    training_match_ids=train_ids,application_match_ids=eval_ids,baseline_grid=list(BASELINE_GRID),grids=[list(g) for g in GRIDS],
    minute_thresholds=MINUTE_THRESHOLDS,min_zone_actions=MIN_ZONE_ACTIONS,bootstrap_replicates=BOOTSTRAP_REPLICATES,seed=SEED,
    focus_keys=FOCUS_KEYS,focus_selection="Top 3 baseline tournament movement-xT totals; retrospective illustration",
    diagnostics={f"{g[0]}x{g[1]}":v for g,v in diagnostics.items()},plots_generated=plt is not None,
    interpretation="Exploratory specification and match sensitivity, not predictive or causal validation")
(REPORT_ROOT / "robustness_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
lines=["# Attacking-value case study: sensitivity and interpretation", "", "The 12×8 historical baseline was reproduced from verified source events. Training uses 64 matches from 2018; application uses 64 matches from 2022. Values describe selected successful passing and carrying contributions, not overall player impact.", "", "## Specification sensitivity", "", "| Grid | Minimum minutes | Eligible players | Spearman vs baseline | Top-10 overlap |", "|---|---:|---:|---:|---:|"]
for r in comparisons:
    rho=f"{r['spearman_vs_baseline']:.3f}" if r['spearman_vs_baseline'] is not None else "NA"
    lines.append(f"| {r['grid']} | {r['min_minutes']} | {r['eligible_players']} | {rho} | {r['top10_overlap']} |")
at300=[r for r in comparisons if r["min_minutes"]==300 and r["grid"]!="12x8"]
if at300:
    lines += ["", "At the 300-minute threshold, alternative grids share "+", ".join(f"{r['top10_overlap']}/10 ({r['grid']})" for r in at300)+" top-10 players with the baseline. Exact shortlist membership is sensitive to grid resolution; a high broad rank correlation does not establish a definitive top ten."]
lines += ["", "## Illustrative players", "", "These players were chosen retrospectively from the top baseline totals. Their differences are not causal effects or established ability differences.", ""]
for r in case_summaries:
    share=f"{r['largest_match_share_of_positive_xt']:.1%}" if r['largest_match_share_of_positive_xt'] is not None else "NA"
    lines += [f"### {r['player']} ({r['team']})", "",f"Across {r['matches']} appearances and {r['minutes']:.1f} elapsed minutes: movement xT {r['movement_xt']:.3f}, passing {r['pass_xt']:.3f}, carrying {r['carry_xt']:.3f}, and {r['movement_xt_per90']:.3f} per 90. There were {r['positive_matches']} positive and {r['negative_matches']} negative match totals. The largest match accounted for {share} of positive movement xT.", "",f"Observed-appearance resampling range (2.5th–97.5th percentiles, fixed model): {r['resampling_lower']:.3f}–{r['resampling_upper']:.3f} per 90. This is descriptive sensitivity, not a calibrated confidence interval. The player qualified in {r['scenario_appearances']} of {len(rankings)} specification/eligibility combinations.", ""]
lines += ["## Limits and next evidence", "", "Grid sensitivity does not test predictive calibration. Match resampling holds the model fixed and ignores many dependencies and selection effects. No inference of overall impact, transfer value or superiority follows from a narrow rank gap. Review the top action IDs and pitch maps with video/context, check role-specific peers, and use a separate untouched cohort if exploratory findings guide model changes.", "", "Data source: StatsBomb Open Data. Follow the supplied source terms and logo requirements before public distribution. Model inspiration: Karun Singh's Expected Threat and socceraction; the explicit variant and attribution scope are documented in notebook 06."]
(REPORT_ROOT / "portfolio_findings.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Saved analysis to",REPORT_ROOT)
